In [ ]:
import ViennaRNA as vrna
import matplotlib.pyplot as plt
from IPython.display import SVG, display
import numpy as np
import pandas as pd

In [ ]:
def complement_DNA(dna_seq):
    dna_seq = dna_seq.replace('A', 't')
    dna_seq = dna_seq.replace('T', 'a')
    dna_seq = dna_seq.replace('C', 'g')
    dna_seq = dna_seq.replace('G', 'c')
    return dna_seq.upper()[::-1]

def complement_RNA(rna_seq):
    rna_seq = dna_seq.replace('A', 'u')
    rna_seq = dna_seq.replace('U', 'a')
    rna_seq = dna_seq.replace('C', 'g')
    rna_seq = dna_seq.replace('G', 'c')
    return rna_seq.upper()[::-1]

def transcribe_DNA(dna_seq):
    dna_seq = dna_seq.upper()
    dna_seq = dna_seq.replace('T', 'U')
    return dna_seq

def combine(list1, list2, sep=""):
    new = []
    for el in list1:
        for el2 in list2:
            new.append(el+sep+el2)
    return(new)

In [ ]:
terminator = "AAACAGATAGGCCCTCttcgGAGGGCCtatctgttTTTTTTTGCTTAATTGTGAGCGCTCACAATT"
seq = transcribe_DNA(terminator)
fc  = vrna.fold_compound(seq)

In [ ]:
fc.mfe()
fc.pf()
fc.MEA()
fc.centroid()
probs = fc.bpp()
entropy = fc.positional_entropy()

In [ ]:
coords = vrna.get_xy_coordinates(fc.MEA()[0])
pairs = []
for pair in vrna.plist(fc.MEA()[0], 0):
    pairs.append([pair.i, pair.j])
    
struct = vrna.plist(fc.MEA()[0], 0)
probs_max = [max(prob) for prob in probs]

for base in struct:
    probs_max[base.i] = max(probs[base.i])
    probs_max[base.j] = max(probs[base.i])

coords_dict = {'seq':seq, 'x':[], 'y':[], 'ent':[], 'bpp':probs_max[1:]}
for idx, letter in enumerate(seq):
    coords_dict['x'].append(coords.get(idx).X)
    coords_dict['y'].append(-coords.get(idx).Y)
    coords_dict['ent'].append(entropy[idx+1]/max(entropy[1:]))

In [ ]:
cmap = plt.colormaps['summer']
color_values = [1- value for value in coords_dict['bpp']]

distances = []
for idx in range(len(seq)):
    if idx!=0:
        dx = coords_dict['x'][idx]-coords_dict['x'][idx-1]
        dy = coords_dict['y'][idx]-coords_dict['y'][idx-1]
        distances.append(np.sqrt(dx**2 + dy**2))
fsize = np.mean(distances)*.5


plt.figure(dpi=200)

#plt.plot(coords_dict['x'][rbs_span[0]:rbs_span[1]], coords_dict['y'][rbs_span[0]:rbs_span[1]], 'bo', markersize=fsize*2)
plt.plot(coords_dict['x'][0], coords_dict['y'][0], 'mo', markersize=fsize*3)

for pair in pairs:
    plt.plot([coords_dict['x'][pair[0]-1], coords_dict['x'][pair[1]-1]], [coords_dict['y'][pair[0]-1], coords_dict['y'][pair[1]-1]], 'r')

plt.plot(coords_dict['x'], coords_dict['y'], color='gray')

for idx in range(len(seq)):
    plt.plot(coords_dict['x'][idx], coords_dict['y'][idx], 'o', markersize=fsize, color=cmap(color_values[idx]))
    plt.text(coords_dict['x'][idx], coords_dict['y'][idx], coords_dict['seq'][idx], horizontalalignment='center', verticalalignment='center', fontsize = fsize)

plt.gca().set_aspect('equal')
plt.axis('off')
plt.show()

---

In [ ]:
terminators = ["T7hyb1", "T7hyb8", "L3S2P11", "L3S1P00", "50bp"]
roadblocks = ["lacO", "tetO", "G3T", "G3Tr", "20bp", "20_lacO", "20_tetO", "20_G3T", "20_G3Tr"]

term_seq = ["AAACAGATAGGCCCTCttcgGAGGGCCtatctgttTTTTTTT",
            "TAAAGAATAAACCttcgGGTTTATTCTTTAGTTTTTTTT",
            "CTCGGTACCAAATTCCAGAAAAGAGACGCTTTCGAGCGTCTTTTTTCGTTTTGGTC",
            "GACGAACAATAAGGGGAGCGGGAAACCGCTCCCCTTTTTTATTGATAACAAAA",
            "CGCTTAACGGCTCCTTACCCGCTTTGCGTTACCTTGCAGGAATCGAGGCC"]

rb_seq = ["AATTGTGAGCGCTCACAATT",
          "TCCCTATCAGTGATAGAGA",
          "GGGTGGGTGGGTGGG",
          "CCCACCCACCCACCC",
          "TATCGGAGCGCCTCCGTACA",
          "TATCGGAGCGCCTCCGTACAAATTGTGAGCGCTCACAATT",
          "TATCGGAGCGCCTCCGTACATCCCTATCAGTGATAGAGA",
          "TATCGGAGCGCCTCCGTACAGGGTGGGTGGGTGGG",
          "TATCGGAGCGCCTCCGTACACCCACCCACCCACCC"]

sticky = "GCTT"

all_samples = combine(terminators, roadblocks, sep="-")
all_seqs = combine(term_seq, rb_seq, sep=sticky)

In [ ]:
def get_RNA_coords(terminator: str):
    # Get the sequence into RNA and calculate ViennaRNA stats
    seq = transcribe_DNA(terminator)
    fc  = vrna.fold_compound(seq)
    mfe = fc.mfe()
    fc.pf()
    fc.MEA()
    fc.centroid()
    probs = fc.bpp()
    entropy = fc.positional_entropy()

    # Generate the coordinates for the fold
    coords = vrna.get_xy_coordinates(fc.MEA()[0])
    pairs = []
    for pair in vrna.plist(fc.MEA()[0], 0):
        pairs.append([pair.i, pair.j])
        
    struct = vrna.plist(fc.MEA()[0], 0)
    probs_max = [max(prob) for prob in probs]
    
    for base in struct:
        probs_max[base.i] = max(probs[base.i])
        probs_max[base.j] = max(probs[base.i])
    
    coords_dict = {'seq':seq, 'x':[], 'y':[], 'ent':[], 'bpp':probs_max[1:]}
    for idx, letter in enumerate(seq):
        coords_dict['x'].append(coords.get(idx).X)
        coords_dict['y'].append(-coords.get(idx).Y)
        coords_dict['ent'].append(entropy[idx+1]/max(entropy[1:]))
    return coords_dict, pairs, mfe

In [ ]:
def plot_coords(coords, pairs_list, ax):
    cmap = plt.colormaps['summer']
    color_values = [1- value for value in coords['bpp']]

    # Calculate distances for connections based on x and y values
    distances = []
    for idx in range(len(coords['seq'])):
        if idx!=0:
            dx = coords['x'][idx]-coords['x'][idx-1]
            dy = coords['y'][idx]-coords['y'][idx-1]
            distances.append(np.sqrt(dx**2 + dy**2))
    fsize = np.mean(distances)*.5

    # Plot sequence start in magenta
    ax.plot(coords['x'][0], coords['y'][0], 'mo', markersize=fsize*3)

    # Plot all connections (hydrogen bonds and covalent) in red
    for pair in pairs_list:
        ax.plot([coords['x'][pair[0]-1], coords['x'][pair[1]-1]], [coords['y'][pair[0]-1], coords['y'][pair[1]-1]], 'r')

    # Plot covalent connections in gray
    ax.plot(coords['x'], coords['y'], color='gray')

    # Plot letters and dots according to bpp
    for idx in range(len(coords['seq'])):
        ax.plot(coords['x'][idx], coords['y'][idx], 'o', markersize=fsize, color=cmap(color_values[idx]))
        ax.text(coords['x'][idx], coords['y'][idx], coords['seq'][idx], horizontalalignment='center', verticalalignment='center', fontsize = fsize)

    ax.tick_params(
        axis='both',          # changes apply to both axes
        which='both',      # both major and minor ticks are affected
        bottom=False,      # ticks along the bottom edge are off
        left=False,         # ticks along the top edge are off
        labelbottom=False,
        labelleft=False)
    ax.set_aspect('equal')
    

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(15, 8), subplot_kw=dict(box_aspect=1), constrained_layout=True)

for i in range(5):
    coords_dict, pairs, mfe = get_RNA_coords(term_seq[i])
    #plot_coords(coords_dict, pairs, axs[i//9, i%9])
    plot_coords(coords_dict, pairs, axs[i])
    axs[i].set_title(terminators[i]+"\nMFE="+str(round(mfe[1], 1)))
#plt.axis('off')
plt.savefig("structure.png")
plt.show()